# Classification benchmark: regex/FlashText vs. zero-shot methods

Scores three tagging methods for the 5 job-category labels against the hand-labeled `gold_is_*` ground truth in `job_tracker.db`:

1. **Baseline** — the regex/FlashText tags already stored in `jobs.is_agent` / `is_ai_llm` / `is_de` / `is_ds` / `is_swe` (produced during scraping, see `src/job_scraper_daily_sql.py`).
2. **Sentence-Transformer zero-shot** — MiniLM embedding cosine similarity between each job description and a short description sentence per category.
3. **NLI zero-shot** — `facebook/bart-large-mnli` via HuggingFace's `zero-shot-classification` pipeline, entailment score per category.

**Task type**: this is **multi-label** classification, not multi-class — a single posting can legitimately be both `is_de` and `is_ds`, or match none of the 5. Each label is scored independently as its own binary problem.

**Scope note**: skips a trained (embedding + sklearn classifier) approach: 255 labeled rows split across 5 independent binary targets — some as sparse as 25 or 51 positives — isn't enough to hold out an honest test split per label, which is exactly why the zero-shot methods are the point of this comparison.

**Metrics**: precision / recall / F1 per label per method (not just accuracy, since labels are imbalanced), plus latency (rows/sec) and model size per method — the speed-vs-accuracy-vs-cost tradeoff table is the actual deliverable, not just "which one wins."

In [1]:
import sqlite3
import time

import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

In [2]:
# Config

DB_PATH = "job_tracker.db"

LABELS = ["is_agent", "is_ai_llm", "is_de", "is_ds", "is_swe"]

# One short description sentence per category, used as the comparison target for
# both zero-shot methods. Phrasing loosely mirrors the keyword lists each regex/
# FlashText category was built from (see setup_keyword_processors in
# src/job_scraper_daily_sql.py) so the zero-shot methods are being asked about the
# same concept the baseline was hand-tuned for.
CATEGORY_DESCRIPTIONS = {
    "is_agent": "building AI agents, agentic workflows, or agentic coding tools like Claude, Cursor, LangChain, AutoGen, or Copilot",
    "is_ai_llm": "working with large language models, generative AI, RAG, prompt engineering, or fine-tuning transformer models",
    "is_de": "data engineering: building ETL/ELT pipelines and data infrastructure with tools like Spark, Airflow, Snowflake, Databricks, or Kafka",
    "is_ds": "data science: machine learning, statistical modeling, or analytics using tools like PyTorch, TensorFlow, scikit-learn, or NLP",
    "is_swe": "general software engineering: backend, frontend, or fullstack development, REST APIs, microservices, Docker, Kubernetes, or CI/CD",
}

ST_THRESHOLD = 0.30   # cosine similarity, all-MiniLM-L6-v2 typically scores short-text pairs in a lower range than long-document pairs
NLI_THRESHOLD = 0.50  # entailment probability from the zero-shot-classification pipeline

## Load labeled data

Pulls `title`, `description`, the baseline `is_*` tags, and the `gold_is_*` ground truth from the `jobs` table, restricted to rows that were actually hand-labeled.

In [3]:
conn = sqlite3.connect(DB_PATH)

gold_cols = ", ".join(f"gold_{c}" for c in LABELS)
base_cols = ", ".join(LABELS)

query = f"""
    SELECT job_id, title, description, {base_cols}, {gold_cols}
    FROM jobs
    WHERE gold_is_agent IS NOT NULL
"""
df = pd.read_sql_query(query, conn)
conn.close()

print(f"Labeled rows: {len(df)}")
for label in LABELS:
    n_pos = int(df[f"gold_{label}"].sum())
    print(f"  gold_{label}: {n_pos} positive / {len(df)} ({n_pos / len(df):.1%})")

Labeled rows: 255
  gold_is_agent: 51 positive / 255 (20.0%)
  gold_is_ai_llm: 82 positive / 255 (32.2%)
  gold_is_de: 107 positive / 255 (42.0%)
  gold_is_ds: 114 positive / 255 (44.7%)
  gold_is_swe: 34 positive / 255 (13.3%)


In [4]:
def score_predictions(y_true: pd.Series, y_pred: pd.Series) -> dict:
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true.astype(int), y_pred.astype(int), average="binary", zero_division=0
    )
    return {"precision": precision, "recall": recall, "f1": f1}

## Method 1: baseline (regex/FlashText)

Recomputing regex/FlashText method to measure performance and latency.

In [ ]:
from flashtext import KeywordProcessor

# Mirrors setup_keyword_processors() in src/job_scraper_daily_sql.py -- keep in sync
# manually if that dict changes, same as the notebook/script duplication
BASELINE_CATEGORIES = {
    'is_agent': ['claude', 'gemini', 'cursor', 'langchain', 'llamaindex', 'autogen', 'crewai', 'agentic', 'devin', 'copilot', 'ai agent', 'ai agents', 'autonomous agent', 'autonomous agents', 'multi-agent', 'agentic workflow', 'langgraph', 'semantic kernel', 'model context protocol', 'tool-calling'],
    'is_ai_llm': ['llm', 'llms', 'generative ai', 'rag', 'fine-tuning', 'prompt engineering', 'transformer', 'vector db', 'openai', 'anthropic', 'genai', 'gpt', 'chatgpt', 'large language model', 'vector database', 'embeddings', 'huggingface', 'bedrock'],
    'is_de': ['etl', 'elt', 'hadoop', 'pyspark', 'spark', 'airflow', 'snowflake', 'databricks', 'bigquery', 'kafka', 'dbt', 'data warehouse', 'data warehousing', 'data lake', 'data pipeline', 'redshift', 'hive', 'flink', 'data engineering'],
    'is_ds': ['machine learning', 'deep learning', 'pytorch', 'tensorflow', 'scikit-learn', 'xgboost', 'nlp', 'power bi', 'tableau', 'looker', 'dashboard', 'business intelligence', 'data visualization', 'forecasting', 'statistical modeling', 'predictive modeling', 'data analysis', 'qlik'],
    'is_swe': ['backend', 'frontend', 'fullstack', 'rest api', 'fastapi', 'django', 'microservices', 'docker', 'kubernetes', 'ci/cd', 'load balancing', 'horizontal scaling', 'autoscaling', 'high availability', 'distributed systems', 'site reliability', 'observability', 'terraform', 'infrastructure as code', 'service mesh', 'fault tolerance', 'production traffic', 'scalability', 'reliability engineering', 'helm'],
    }

baseline_setup_start = time.perf_counter()
baseline_processors = {}
for col_name, keywords in BASELINE_CATEGORIES.items():
    kp = KeywordProcessor(case_sensitive=False)
    for kw in keywords:
        kp.add_keyword(kw)
    baseline_processors[col_name] = kp
baseline_setup_seconds = time.perf_counter() - baseline_setup_start
print(f"Processor setup: {baseline_setup_seconds:.4f}s")

combined_text = df["title"].fillna("") + " " + df["description"].fillna("")

new_result_dict = {}
baseline_infer_start = time.perf_counter()
for col_name, kp in baseline_processors.items():
    new_result_dict[col_name] = combined_text.apply(lambda text: bool(kp.extract_keywords(text)))
baseline_infer_seconds = time.perf_counter() - baseline_infer_start

baseline_throughput = len(df) / baseline_infer_seconds
print(f"Inference: {baseline_infer_seconds:.4f}s for {len(df)} rows ({baseline_throughput:.1f} rows/sec)")

baseline_rows = []
for label in BASELINE_CATEGORIES.keys():
    metrics = score_predictions(df[f"gold_{label}"], new_result_dict[label])
    baseline_rows.append({"method": "baseline_regex", "label": label, **metrics})

baseline_results = pd.DataFrame(baseline_rows)
baseline_results

Processor setup: 0.0005s
Inference: 1.1116s for 255 rows (229.4 rows/sec)


,method,label,precision,recall,f1
0,baseline_regex,is_agent,0.786885,0.941176,0.857143
1,baseline_regex,is_ai_llm,0.880435,0.987805,0.931034
2,baseline_regex,is_de,0.822034,0.906542,0.862222
3,baseline_regex,is_ds,0.604520,0.938596,0.735395
4,baseline_regex,is_swe,0.351648,0.941176,0.512000


## Method 2: Sentence-Transformer zero-shot (MiniLM cosine similarity)

Embeds each job description and each category description sentence with `all-MiniLM-L6-v2`, then thresholds cosine similarity per label. Timed end-to-end (model load excluded from the per-row latency figure, reported separately) so the latency number reflects steady-state inference throughput, not one-time startup cost.

In [22]:
from sentence_transformers import SentenceTransformer, util

st_load_start = time.perf_counter()
st_model = SentenceTransformer("all-MiniLM-L6-v2")
st_load_seconds = time.perf_counter() - st_load_start

st_param_count = sum(p.numel() for p in st_model._first_module().auto_model.parameters())
st_model_size_mb = st_param_count * 4 / (1024 ** 2)  # fp32 parameters

print(f"Model load: {st_load_seconds:.2f}s, {st_param_count:,} params (~{st_model_size_mb:.0f} MB fp32)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model load: 6.02s, 22,713,216 params (~87 MB fp32)


In [ ]:
category_labels_ordered = list(CATEGORY_DESCRIPTIONS.keys())
category_sentences = [CATEGORY_DESCRIPTIONS[label] for label in category_labels_ordered]
category_embeddings = st_model.encode(category_sentences, convert_to_tensor=True)

st_infer_start = time.perf_counter()
description_embeddings = st_model.encode(
    df["description"].fillna("").tolist(), convert_to_tensor=True, show_progress_bar=True
)
similarity_matrix = util.cos_sim(description_embeddings, category_embeddings).cpu().numpy()
st_infer_seconds = time.perf_counter() - st_infer_start

st_throughput = len(df) / st_infer_seconds
print(f"Inference: {st_infer_seconds:.2f}s for {len(df)} rows ({st_throughput:.1f} rows/sec)")

st_scores = pd.DataFrame(similarity_matrix, columns=category_labels_ordered, index=df.index)


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Inference: 7.78s for 255 rows (32.8 rows/sec)


In [24]:
for label in LABELS:
    pos = st_scores[label][df[f"gold_{label}"] == 1]
    neg = st_scores[label][df[f"gold_{label}"] == 0]
    print(f"{label}: positives mean={pos.mean():.3f}, negatives mean={neg.mean():.3f}, gap={pos.mean()-neg.mean():.3f}")

is_agent: positives mean=0.432, negatives mean=0.292, gap=0.140
is_ai_llm: positives mean=0.354, negatives mean=0.233, gap=0.121
is_de: positives mean=0.496, negatives mean=0.323, gap=0.172
is_ds: positives mean=0.360, negatives mean=0.319, gap=0.041
is_swe: positives mean=0.352, negatives mean=0.253, gap=0.099


In [ ]:
st_predictions = (st_scores >= ST_THRESHOLD).astype(int)
st_rows = []
for label in LABELS:
    metrics = score_predictions(df[f"gold_{label}"], st_predictions[label])
    st_rows.append({"method": "sentence_transformer", "label": label, **metrics})

st_results = pd.DataFrame(st_rows)
st_results

,method,label,precision,recall,f1
0,sentence_transformer,is_agent,0.338346,0.882353,0.489130
1,sentence_transformer,is_ai_llm,0.650000,0.792683,0.714286
2,sentence_transformer,is_de,0.523077,0.953271,0.675497
3,sentence_transformer,is_ds,0.514793,0.763158,0.614841
4,sentence_transformer,is_swe,0.244186,0.617647,0.350000


## Method 2.2 Sentence-Transformer zero-shot (NomicAI cosine similarity)

Embeds each job description and each category description sentence with `nomic-ai/nomic-embed-text-v1.5` (8,192-token context), then thresholds cosine similarity per label — same method as Method 2, different model.

**Motivation**: `all-MiniLM-L6-v2`'s 256-token limit is a real constraint here, not a theoretical one — a separate check against this dataset found 93.8% of job descriptions exceed 256 tokens (median 676). Method 2 was likely scoring against roughly the first third of most postings. This tests whether removing that truncation actually improves the Sentence-Transformer approach, or whether the original low score was driven by something else (e.g. generic category-description phrasing) that truncation wasn't the main cause of.

In [27]:
from sentence_transformers import SentenceTransformer, util

nomicai_load_start = time.perf_counter()
nomicai_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
nomicai_load_seconds = time.perf_counter() - nomicai_load_start

nomicai_param_count = sum(p.numel() for p in nomicai_model._first_module().auto_model.parameters())
nomicai_model_size_mb = nomicai_param_count * 4 / (1024 ** 2)  # fp32 parameters

print(f"Model load: {nomicai_load_seconds:.2f}s, {nomicai_param_count:,} params (~{nomicai_model_size_mb:.0f} MB fp32)")

<All keys matched successfully>


Model load: 9.94s, 136,731,648 params (~522 MB fp32)


In [28]:
category_labels_ordered = list(CATEGORY_DESCRIPTIONS.keys())
category_sentences = [CATEGORY_DESCRIPTIONS[label] for label in category_labels_ordered]
category_embeddings = nomicai_model.encode(category_sentences, convert_to_tensor=True)

nomicai_infer_start = time.perf_counter()
description_embeddings = nomicai_model.encode(
    df["description"].fillna("").tolist(),
    convert_to_tensor=True, show_progress_bar=True,
)
similarity_matrix = util.cos_sim(description_embeddings, category_embeddings).cpu().numpy()
nomicai_infer_seconds = time.perf_counter() - nomicai_infer_start

nomicai_throughput = len(df) / nomicai_infer_seconds
print(f"Inference: {nomicai_infer_seconds:.2f}s for {len(df)} rows ({nomicai_throughput:.1f} rows/sec)")

nomicai_scores = pd.DataFrame(similarity_matrix, columns=category_labels_ordered, index=df.index)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Inference: 301.68s for 255 rows (0.8 rows/sec)


In [29]:
for label in LABELS:
    pos = nomicai_scores[label][df[f"gold_{label}"] == 1]
    neg = nomicai_scores[label][df[f"gold_{label}"] == 0]
    print(f"{label}: positives mean={pos.mean():.3f}, negatives mean={neg.mean():.3f}, gap={pos.mean()-neg.mean():.3f}")

is_agent: positives mean=0.646, negatives mean=0.552, gap=0.094
is_ai_llm: positives mean=0.630, negatives mean=0.560, gap=0.070
is_de: positives mean=0.663, negatives mean=0.592, gap=0.071
is_ds: positives mean=0.621, negatives mean=0.572, gap=0.049
is_swe: positives mean=0.599, negatives mean=0.535, gap=0.064


In [30]:
nomicai_predictions = (nomicai_scores >= 0.58).astype(int)

nomicai_rows = []
for label in LABELS:
    metrics = score_predictions(df[f"gold_{label}"], nomicai_predictions[label])
    nomicai_rows.append({"method": "nomic_ai_sentence_transformer", "label": label, **metrics})

nomicai_results = pd.DataFrame(nomicai_rows)
nomicai_results

,method,label,precision,recall,f1
0,nomic_ai_sentence_transformer,is_agent,0.421053,0.941176,0.581818
1,nomic_ai_sentence_transformer,is_ai_llm,0.565574,0.841463,0.676471
2,nomic_ai_sentence_transformer,is_de,0.492537,0.925234,0.642857
3,nomic_ai_sentence_transformer,is_ds,0.583333,0.798246,0.674074
4,nomic_ai_sentence_transformer,is_swe,0.409091,0.529412,0.461538


## Method 3: NLI zero-shot (`facebook/bart-large-mnli`)

Uses HuggingFace's `zero-shot-classification` pipeline with `multi_label=True` (each candidate label scored independently against the premise, which is what multi-label needs -- `multi_label=False` would normalize scores to sum to 1 across labels, which is wrong here since a posting can match several categories at once). This is the method expected to do better specifically on contradiction/negation framing, since entailment/contradiction is a trained NLI objective rather than something cosine similarity picks up.

In [ ]:
from transformers import pipeline

nli_load_start = time.perf_counter()
nli_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
nli_load_seconds = time.perf_counter() - nli_load_start

nli_param_count = sum(p.numel() for p in nli_classifier.model.parameters())
nli_model_size_mb = nli_param_count * 4 / (1024 ** 2)  # fp32 parameters

print(f"Model load: {nli_load_seconds:.2f}s, {nli_param_count:,} params (~{nli_model_size_mb:.0f} MB fp32)")

Model load: 54.35s, 407,344,131 params (~1554 MB fp32)


In [9]:
HYPOTHESIS_TEMPLATE = "This job posting involves {}."

nli_infer_start = time.perf_counter()
nli_raw_results = nli_classifier(
    df["description"].fillna("").tolist(),
    candidate_labels=category_sentences,
    hypothesis_template=HYPOTHESIS_TEMPLATE,
    multi_label=True,
)
nli_infer_seconds = time.perf_counter() - nli_infer_start

nli_throughput = len(df) / nli_infer_seconds
print(f"Inference: {nli_infer_seconds:.2f}s for {len(df)} rows ({nli_throughput:.1f} rows/sec)")

# nli_raw_results is a list of dicts (one per row), each with parallel 'labels'/'scores'
# lists NOT guaranteed to be in category_sentences order (the pipeline sorts by score) --
# realign to CATEGORY_DESCRIPTIONS order before building the score matrix.
sentence_to_label = {v: k for k, v in CATEGORY_DESCRIPTIONS.items()}
nli_scores = pd.DataFrame(
    [
        {sentence_to_label[lbl]: score for lbl, score in zip(row["labels"], row["scores"])}
        for row in nli_raw_results
    ],
    index=df.index,
)[category_labels_ordered]
nli_predictions = (nli_scores >= NLI_THRESHOLD).astype(int)

Inference: 4134.81s for 255 rows (0.1 rows/sec)


In [10]:
nli_rows = []
for label in LABELS:
    metrics = score_predictions(df[f"gold_{label}"], nli_predictions[label])
    nli_rows.append({"method": "nli_zero_shot", "label": label, **metrics})

nli_results = pd.DataFrame(nli_rows)
nli_results

,method,label,precision,recall,f1
0,nli_zero_shot,is_agent,0.265896,0.901961,0.410714
1,nli_zero_shot,is_ai_llm,0.370892,0.963415,0.535593
2,nli_zero_shot,is_de,0.514706,0.981308,0.675241
3,nli_zero_shot,is_ds,0.459016,0.982456,0.625698
4,nli_zero_shot,is_swe,0.172043,0.941176,0.290909


## Combined tradeoff table

Per-label precision/recall/F1 for all three methods, plus a separate cost/latency/size summary per method (latency and model size are method-level, not per-label, so they're kept in a second table rather than repeated on every row).

In [11]:
all_results = pd.concat([baseline_results, st_results, nli_results], ignore_index=True)
pivoted = all_results.pivot(index="label", columns="method", values=["precision", "recall", "f1"])
pivoted

precision                                            recall  \
method    baseline_regex nli_zero_shot sentence_transformer baseline_regex   
label                                                                        
is_agent        0.750000      0.265896             0.338346       0.764706   
is_ai_llm       0.903614      0.370892             0.650000       0.914634   
is_de           0.871287      0.514706             0.523077       0.822430   
is_ds           0.672269      0.459016             0.514793       0.701754   
is_swe          0.477612      0.172043             0.244186       0.941176   

                                                         f1                \
method    nli_zero_shot sentence_transformer baseline_regex nli_zero_shot   
label                                                                       
is_agent       0.901961             0.882353       0.757282      0.410714   
is_ai_llm      0.963415             0.792683       0.909091      0.535593   
is_de          0.981308             0.953271       0.846154      0.675241   
is_ds          0.982456             0.763158       0.686695      0.625698   
is_swe         0.941176             0.617647       0.633663      0.290909   

                                
method    sentence_transformer  
label                           
is_agent              0.489130  
is_ai_llm             0.714286  
is_de                 0.675497  
is_ds                 0.614841  
is_swe                0.350000

In [14]:
cost_summary = pd.DataFrame(
    [
        {
            "method": "baseline_regex",
            "model_size_mb": 0.0,
            "throughput_rows_per_sec": baseline_throughput,  # precomputed at scrape time, not measured here
        },
        {
            "method": "sentence_transformer",
            "model_size_mb": st_model_size_mb,
            "throughput_rows_per_sec": st_throughput,
        },
        {
            "method": "nli_zero_shot",
            "model_size_mb": nli_model_size_mb,
            "throughput_rows_per_sec": nli_throughput,
        },
    ]
)
macro_f1 = all_results.groupby("method")["f1"].mean().rename("macro_f1")
cost_summary = cost_summary.merge(macro_f1, on="method")
cost_summary

,method,model_size_mb,throughput_rows_per_sec,macro_f1
0,baseline_regex,0.000000,217.190001,0.766577
1,sentence_transformer,86.644043,31.878749,0.568751
2,nli_zero_shot,1553.894543,0.061672,0.507631


## Notes / caveats

- Thresholds (`ST_THRESHOLD`, `NLI_THRESHOLD`) are fixed defaults, not tuned -- with only ~255 labeled rows there's no clean way to hold out a separate validation split for threshold-tuning without shrinking the test set this notebook scores against. If either zero-shot method looks competitive, tuning the threshold (e.g. sweeping and picking the best F1) is the natural next step before trusting these numbers further.
- `macro_f1` in the cost table averages F1 equally across the 5 labels regardless of support -- `is_swe` (25 positives) and `is_ds` (106 positives) count the same. That's deliberate here (all 5 categories matter equally for tagging purposes), but worth stating explicitly since it's a real modeling choice, not a neutral default.